# DINOv2 ViT-S/14 — DIMER image feature extraction tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/dinov2-feature-extraction-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/dinov2-feature-extraction-pipeline/blob/main/tutorials/dinov2_feature_extraction_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-timm%2Fvit__small__patch14__dinov2.lvd142m-ffcc4d?style=flat)](https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m)
[![Upstream](https://img.shields.io/badge/Upstream-facebookresearch%2Fdinov2-181717?style=flat&logo=github&logoColor=white)](https://github.com/facebookresearch/dinov2)
[![arXiv](https://img.shields.io/badge/arXiv-2304.07193-b31b1b.svg)](https://arxiv.org/abs/2304.07193)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** self-supervised image feature extraction (one 384-d embedding per image) using the pinned DINOv2 ViT-S/14 `lvd142m` weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`DINOv2FeatureExtractionPipeline`) rather than reimplementing model inference. At inference each image is resized so its shorter side is 518 px, centre-cropped to 518 x 518, normalised with the ImageNet mean/std from the snapshot config, and passed through the ViT-S/14 backbone with no classifier head; the class token after the final LayerNorm is taken as the image's feature (`POOLING = "cls"`) and L2-normalised (`NORMALIZED = True`), so a dot product between two vectors is their cosine similarity. **Embeddings are representations, not predictions:** the pipeline returns no label, no score, no class and no threshold, and no intrinsic accuracy exists for a vector on its own. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. What the upstream checkpoint supplies is the architecture, weights and preprocessing config; what this repository adds is manifest verification, input validation, the class-token pooling choice, L2 normalisation, and a fixed output contract.

**Learning objectives:** bootstrap the repository in a fresh runtime, generate a synthetic default input set (or upload your own), surface the pipeline's ceilings and the embedding contract, stage and digest-verify the immutable upstream snapshot, embed a small batch through the public API, read the embedding output (shape, unit, pooling, normalisation) correctly, run a qualitative cosine-similarity check and understand why it is not a metric, and export vectors with their identifiers plus provenance.

**This notebook does not demonstrate:** image classification, object detection, segmentation, captioning, image-text comparison, patch-level (dense) features, attention maps, or any similarity search, clustering or probe — the repository exposes only the pooled per-image vector; everything downstream is the caller's code.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. Each 518 px image costs about 46.8 GMACs (upstream card), so a three-image CPU batch takes seconds to tens of seconds on a hosted CPU runtime; on the model card's GPU (RTX 5070 Ti) the verified snapshot loaded in 3.4 s and a two-image batch took 1.2 s. The pinned `torch==2.14.0` install and the 88 MB checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python and NumPy; what an embedding vector is and why cosine similarity between two vectors is not an accuracy.
- **Data:** the default sample is a set of three synthetic images generated in code; BYOD is one or more image files, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded images remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing snapshot file from the Hugging Face Hub at the immutable revision. No credentials are needed.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `torchvision`, `timm`, `huggingface-hub`, `safetensors`, `numpy`, `pillow`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CUDA when available, otherwise on CPU; no half precision, compilation, or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/dinov2-feature-extraction-pipeline.git'
REPO_NAME = 'dinov2-feature-extraction-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, PIL, numpy, timm, torch
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'numpy': numpy.__version__, 'pillow': PIL.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample set or optional BYOD

The default sample is **synthetic**: three 256 x 256 RGB images built in code — a deterministic gradient (red ramps left to right, green top to bottom, blue is their mean), the same gradient rotated by 180 degrees, and a flat mid-grey block. They need no download, contain no personal data, and each one's pixel SHA-256 is printed and exported alongside its identifier; no randomness is involved, so no seed is needed. None of them is a photograph, so the embeddings they produce are smoke/sanity evidence that the code path works: the rotated copy is there only so that Section 5 can show a *qualitative* similarity comparison (a near-duplicate versus an unrelated image), which is not a benchmark and not a metric.

BYOD is optional and disabled by default. Expected BYOD input: one or more image files that Pillow can open (PNG, JPEG, WebP, ...), any mode (converted to RGB), each with both sides between 1 and `MAX_IMAGE_SIDE` = 4096 px, at most `MAX_BATCH` files per run; each will be resized to shorter side 518 and centre-cropped to 518 x 518. The uploads stay inside this runtime.

In [ ]:
import hashlib
import io

import numpy as np
from PIL import Image

USE_BYOD = False  # @param {type:"boolean"}
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    images = {}
    for name, data in uploaded.items():
        image = Image.open(io.BytesIO(data))
        image.load()
        images[name] = image
    sample_kind = 'BYOD upload'
else:
    # Deterministic synthetic set: no randomness, so no seed is needed and the digests are stable.
    side = 256
    red = np.tile(np.linspace(0.0, 255.0, side), (side, 1))
    green = red.T
    blue = (red + green) / 2.0
    gradient = Image.fromarray(np.rint(np.stack([red, green, blue], axis=-1)).astype(np.uint8))
    images = {
        'synthetic_gradient_256': gradient,
        'synthetic_gradient_256_rot180': gradient.rotate(180),
        'synthetic_flat_grey_256': Image.new('RGB', (side, side), (128, 128, 128)),
    }
    sample_kind = 'synthetic (generated in this cell)'
image_ids = list(images)
digests = {name: hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest() for name, image in images.items()}
for name in image_ids:
    print({'id': name, 'mode': images[name].mode, 'size': images[name].size, 'pixel_sha256': digests[name]})
print({'sample_kind': sample_kind, 'count': len(image_ids)})

## 3. Validate the inputs against the pipeline ceilings and state the embedding contract

The pipeline enforces two operational ceilings and fixes three contract constants, all imported here from the package so the values shown are the ones in force: `MAX_IMAGE_SIDE` (either side, px) and `MAX_BATCH` (images per `embed` call); `EMBED_DIM` (vector length, 384), `POOLING` (`"cls"`: the class token after the final LayerNorm — one vector **per image**, not per patch or per token) and `NORMALIZED` (`True`: every vector has unit L2 norm). This cell surfaces them and checks the batch before any model work, naming the failing condition and the corrective action; `embed()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. **What the pipeline changes about your images:** each is converted to RGB, resized so its shorter side is 518 px and centre-cropped to 518 x 518 (`crop_pct: 1.0`, `crop_mode: "center"` in the snapshot config) — for a non-square image the outer strips along the longer side never reach the encoder; nothing else is dropped, and there is no missing-data concept: every pixel of the cropped square is visible to the encoder. The notebook itself does not resize, crop, or subsample.

In [ ]:
from dinov2_feature_extraction_pipeline import EMBED_DIM, MAX_BATCH, MAX_IMAGE_SIDE, NORMALIZED, POOLING

ceilings = {'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_BATCH': MAX_BATCH}
contract = {'EMBED_DIM': EMBED_DIM, 'POOLING': POOLING, 'NORMALIZED': NORMALIZED, 'unit': 'one vector per image'}
print(ceilings)
print(contract)
problems = []
if not 1 <= len(image_ids) <= MAX_BATCH:
    problems.append(f'{len(image_ids)} images is outside 1..MAX_BATCH={MAX_BATCH}: upload fewer files or split the batch')
for name in image_ids:
    width, height = images[name].size
    if width < 1 or height < 1 or max(width, height) > MAX_IMAGE_SIDE:
        problems.append(f'{name}: image side {images[name].size} outside 1..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: resize the image and rerun Section 2')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'batch_size': len(image_ids), 'model_input': '518x518 (shorter side resized to 518, centre crop)', 'within_ceilings': True})

## 4. Stage, verify, and resolve the pinned model

The public API pins the exact upstream model repository and immutable 40-hex revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here); timm executes no remote code and the Hub is used only at that revision. The repository commits the DIMER snapshot manifest (`weights/vit-small-dinov2/dimer-base-manifest.json`: model id, revision, and the byte size and SHA-256 of each snapshot file) but git-ignores the 88 MB `model.safetensors`, so a fresh clone must stage the missing file first. The package's `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches only the manifest-listed files that are absent, from the Hub at the pinned revision, into the repository's weights directory, and returns the list it fetched (`['model.safetensors']` on a fresh clone, `[]` when everything is already staged); it refuses to stage if the committed manifest disagrees with the package's pinned identity. `verify_snapshot()` then re-hashes every listed file and raises on the first size or digest mismatch, and only afterwards does `from_pretrained` build the headless backbone from that verified directory (`source: local-snapshot`), additionally checking that the model's feature width equals `EMBED_DIM`. The effective model identity, the resolved input size, and the selected device are printed before inference.

In [ ]:
from dinov2_feature_extraction_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, DINOv2FeatureExtractionPipeline, stage_missing_files, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
# Only the manifest-listed files that are absent are fetched, at the immutable revision the
# package pins; verify_snapshot then checks every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'fetched': fetched, 'from': MODEL_ID, 'revision': MODEL_REVISION})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
print({'snapshot_path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')})
pipe = DINOv2FeatureExtractionPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'source': pipe.source, 'input_size': pipe.input_size, 'precision': 'float32'})

## 5. Embed the batch and run a qualitative similarity check

`embed` returns `embeddings` (a list with one 384-float list per input, **in input order**, so position *i* belongs to `image_ids[i]`), plus `dim`, `pooling`, `normalized`, `input_size`, `device`, `source`, `model_id` and `model_revision`. The sanity checks below are falsifiable plumbing checks (one vector per input, length `EMBED_DIM`, finite, unit norm within float tolerance); they are not a quality measure.

**Evaluation:** the repository ships **no metric helper and reports no performance measure**, because an embedding is a representation, not a prediction — there is nothing to score it against on its own. Evaluating these features requires a downstream labelled task: for example a retrieval set with relevance labels (mean average precision), a labelled image set for a linear probe or k-NN classifier (accuracy), or human-judged duplicate pairs for a similarity threshold; the DINOv2 paper evaluates exactly such tasks, and those upstream numbers are not measured here. The pairwise cosine similarities printed below (a dot product, because the vectors are unit-normalised) are a **qualitative check only**: the rotated copy is expected to score higher against the original than the flat block does, which shows that the vector encodes image content, but the absolute values mean nothing without a threshold calibrated on your own labelled pairs, and the pipeline deliberately ships none. On synthetic non-photographic inputs the numbers are plumbing evidence, not benchmark evidence. Small numeric differences between CPU and CUDA kernels move each vector slightly, so similarities are repeatable on fixed hardware but not bitwise-identical across devices. The runtime figure is measured on the runtime identified in Section 1 for this batch and includes the first-call warm-up.

In [ ]:
import time

started = time.perf_counter()
result = pipe.embed([images[name] for name in image_ids])
elapsed = time.perf_counter() - started
vectors = np.asarray(result['embeddings'], dtype=np.float32)
norms = np.linalg.norm(vectors, axis=1)
checks = {
    'one_vector_per_input': vectors.shape[0] == len(image_ids),
    'dim_matches_contract': vectors.shape[1] == result['dim'] == EMBED_DIM,
    'all_finite': bool(np.isfinite(vectors).all()),
    'unit_norm': bool(np.allclose(norms, 1.0, atol=1e-4)),
}
if not all(checks.values()):
    raise RuntimeError(f'embedding output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'embeddings'})
print({'embeddings_shape': list(vectors.shape), 'norms': [round(float(n), 6) for n in norms], 'seconds': round(elapsed, 3), 'checks': checks})
cosine = vectors @ vectors.T
print('pairwise cosine similarity (qualitative check only; no threshold is shipped):')
for i, name in enumerate(image_ids):
    print(f"{name:>32}  " + '  '.join(f'{cosine[i, j]:.4f}' for j in range(len(image_ids))))
metrics = {}
print('no metric is reported: embeddings are representations, not predictions; evaluate them on a downstream labelled task')

## 6. Export vectors, identifiers, and provenance

One JSON record is written under `outputs/`: an `items` list with, per image, its identifier, pixel digest, size and 384-float vector (so every vector maps back to its input), the embedding contract (`dim`, `pooling`, `normalized`, `input_size`), the pairwise cosine matrix keyed by the same identifiers, the sanity checks, the ceilings in force, the empty metric block, the repository revision, the model identifier and immutable revision, the verified snapshot summary, and the runtime identity (Python, `torch`, `timm`, NumPy, Pillow, device, precision). No credentials are involved in any step, so none can reach the export.

In [ ]:
import json

os.makedirs('outputs', exist_ok=True)
payload = {
    'items': [
        {'id': name, 'pixel_sha256': digests[name], 'width': images[name].width, 'height': images[name].height, 'vector': result['embeddings'][index]}
        for index, name in enumerate(image_ids)
    ],
    'dim': result['dim'],
    'pooling': result['pooling'],
    'normalized': result['normalized'],
    'input_size': result['input_size'],
    'cosine_similarity': {'ids': image_ids, 'matrix': [[round(float(value), 6) for value in row] for row in cosine]},
    'sanity_checks': checks,
    'ceilings': ceilings,
    'metrics': metrics,
    'sample_kind': sample_kind,
    'seconds': round(elapsed, 3),
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'snapshot': {'path': snapshot_info['path'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')},
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'timm': timm.__version__,
        'numpy': numpy.__version__,
        'pillow': PIL.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/dinov2_feature_extraction_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/dinov2_feature_extraction_result.json')

## Interpretation and limits

Each output is a 384-d unit vector: a representation of the image, not a prediction — no label, no score, no probability, no threshold. Cosine similarity between two vectors is meaningful only relative to other pairs from the same model and must be calibrated on labelled pairs before it can decide anything; the vector encodes style, layout and background as well as subject, so near-duplicates and merely similar-looking images can score alike. On the synthetic sample set the similarities are plumbing evidence only, and no metric is reported because none exists without a downstream labelled task. Inputs are centre-cropped to 518 x 518 after resizing the shorter side, so content near the long edges of non-square images does not reach the encoder; the pipeline exposes only the pooled class token (no patch features, attention maps or register tokens), does not classify, detect, segment or compare to text, and does not detect out-of-distribution inputs. Inference is deterministic given the same weights, device and library versions; results on fixed hardware are repeatable but not guaranteed bitwise-identical across devices. The upstream weights are Apache-2.0 (DINOv2 was re-licensed from CC-BY-NC-4.0 on 2023-08-31; the stale `cc-by-nc-4.0` field in the snapshot `config.json` does not govern them) — see `../docs/WEIGHTS.md` for the provenance record.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify the pinned model snapshot, validate the demonstrated inputs against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, retrieval or probe quality on any domain, a usable similarity threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/vit-small-dinov2/` and rerun Section 4. A `ValueError` naming `MAX_BATCH` or `MAX_IMAGE_SIDE` in Section 3: upload fewer or smaller files and rerun from Section 2. A slow Section 5 on a CPU runtime is expected at 46.8 GMACs per image; a CUDA runtime is used automatically when present.

**Next experiments.** Upload a handful of real photographs with `USE_BYOD` enabled — including two views of the same object — and inspect whether the pairwise cosine ranking matches your judgement; build a tiny labelled retrieval set and compute mean average precision from the exported vectors to see what a real evaluation needs; compare the CUDA and CPU cosine matrices on the same batch to observe kernel-level variability. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance and license record: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/timm/vit_small_patch14_dinov2.lvd142m
- Upstream code: https://github.com/facebookresearch/dinov2
- DINOv2 paper: https://arxiv.org/abs/2304.07193
- timm documentation: https://huggingface.co/docs/timm